# HW2D — Run, Plot, and Animate

This notebook runs a Hasegawa–Wakatani 2D turbulence simulation (or loads an existing one), then produces:
- **Scalar time-series plots** — energy, enstrophy, fluxes
- **Field snapshots** — density *n*, vorticity *ω*, potential *φ* at multiple time points
- **Animated video** of the evolving 2D fields, with an option to save it to disk

In [ ]:
# ── Configuration ────────────────────────────────────────────────────────────
import sys, pathlib
ROOT = pathlib.Path(".").resolve().parent   # hw2d_ssm/ → repo root
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

# ── Simulation parameters ────────────────────────────────────────────────────
# Switch USE_EXISTING_DATA to True (and set H5_PATH) to skip re-running
USE_EXISTING_DATA = True
H5_PATH = ROOT / "hw2d_ssm" / "hw2d_ssm" / "data" / "hw2d_medium.h5"

# Short demo run — comment out / lower end_time for a quick test
DEMO_END_TIME = 200.0       # 200 → ~1–2 min wall time on M4 Pro
DEMO_GRID     = 64          # smaller grid → much faster
DEMO_SNAPS    = 10          # frame_dt = step_size * snaps = 0.25
DEMO_OUT      = ROOT / "hw2d_ssm" / "hw2d_ssm" / "data" / "hw2d_demo.h5"

# ── Video / output settings ───────────────────────────────────────────────────
OUTPUT_DIR   = ROOT / "hw2d_ssm" / "outputs"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

VIDEO_PATH   = OUTPUT_DIR / "hw2d_fields.mp4"   # change to .gif if no ffmpeg
SAVE_VIDEO   = True         # set False to skip saving
VIDEO_FPS    = 20
VIDEO_DPI    = 100

# Frames to animate (stride to keep the video short for large datasets)
ANIM_FRAME_STRIDE = 1       # 1 = every frame; 5 = every 5th frame, etc.

# Drop transient startup frames before plotting scalars
DROP_INITIAL = 50

print(f"ROOT      : {ROOT}")
print(f"OUTPUT_DIR: {OUTPUT_DIR}")
print(f"VIDEO_PATH: {VIDEO_PATH}")

In [ ]:
# ── Imports ───────────────────────────────────────────────────────────────────
import h5py
import numpy as np
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.animation as animation
from IPython.display import HTML, display

matplotlib.rcParams.update({
    "figure.dpi": 110,
    "axes.titlesize": 11,
    "axes.labelsize": 10,
})
print("imports ok")

## 1 — Run or Load Simulation

In [ ]:
from hw2d_ssm.data.simulate import HWConfig, run_hw2d

if USE_EXISTING_DATA and H5_PATH.exists():
    h5_path = H5_PATH
    print(f"Using existing data: {h5_path}")
else:
    print("Running demo simulation (set USE_EXISTING_DATA=True to skip) …")
    cfg = HWConfig(
        grid_pts   = DEMO_GRID,
        end_time   = DEMO_END_TIME,
        step_size  = 0.025,
        snaps      = DEMO_SNAPS,
        c1         = 1.0,
        seed       = 42,
        output_path= str(DEMO_OUT),
    )
    h5_path = run_hw2d(cfg, force_recompute=False)

print(f"HDF5 size: {h5_path.stat().st_size / 1e6:.1f} MB")

In [ ]:
# ── Load all data from HDF5 ───────────────────────────────────────────────────
with h5py.File(h5_path, "r") as hf:
    density    = np.asarray(hf["density"],  dtype=np.float32)   # (T, y, x)
    omega      = np.asarray(hf["omega"],    dtype=np.float32)
    phi        = np.asarray(hf["phi"],      dtype=np.float32)
    energy     = np.asarray(hf["energy"],   dtype=np.float32).ravel()
    enstrophy  = np.asarray(hf["enstrophy"],dtype=np.float32).ravel()

    # load optional scalars if present
    def _load_scalar(hf, key):
        return np.asarray(hf[key], dtype=np.float32).ravel() if key in hf else None

    gamma_n = _load_scalar(hf, "gamma_n")
    gamma_c = _load_scalar(hf, "gamma_c")

    attrs      = dict(hf.attrs)
    frame_dt   = float(attrs.get("frame_dt", attrs.get("dt", 1.0)))

T = density.shape[0]
times = np.arange(T) * frame_dt

print(f"T={T} frames, frame_dt={frame_dt:.4f}, total_time={times[-1]:.1f}")
print(f"Grid: {density.shape[1]}×{density.shape[2]}")
print(f"Attributes: { {k: v for k, v in attrs.items()} }")

## 2 — Scalar Time Series

In [ ]:
drop = min(DROP_INITIAL, T // 4)
t_plot  = times[drop:]
E_plot  = energy[drop:]
O_plot  = enstrophy[drop:]

n_scalar_rows = 2 + (gamma_n is not None) + (gamma_c is not None)
fig, axes = plt.subplots(n_scalar_rows, 1, figsize=(10, 2.5 * n_scalar_rows), sharex=True)

ax_iter = iter(axes)

ax = next(ax_iter)
ax.plot(t_plot, E_plot, lw=1.0, color="steelblue")
ax.set_ylabel("Energy")
ax.set_title("HW2D scalar diagnostics")

ax = next(ax_iter)
ax.plot(t_plot, O_plot, lw=1.0, color="darkorange")
ax.set_ylabel("Enstrophy")

if gamma_n is not None:
    ax = next(ax_iter)
    ax.plot(t_plot, gamma_n[drop:], lw=1.0, color="green")
    ax.set_ylabel(r"$\Gamma_n$")

if gamma_c is not None:
    ax = next(ax_iter)
    ax.plot(t_plot, gamma_c[drop:], lw=1.0, color="purple")
    ax.set_ylabel(r"$\Gamma_c$")

axes[-1].set_xlabel("Time")
fig.tight_layout()
scalar_fig_path = OUTPUT_DIR / "hw2d_scalars.png"
fig.savefig(scalar_fig_path, dpi=130, bbox_inches="tight")
print(f"Saved: {scalar_fig_path}")
plt.show()

## 3 — 2D Field Snapshots

In [ ]:
# Pick a handful of evenly-spaced snapshot times
N_SNAPS    = 5
snap_idxs  = np.linspace(drop, T - 1, N_SNAPS, dtype=int)

FIELDS = [
    ("density",   density,  "RdBu_r",  r"Density $n$"),
    ("omega",     omega,    "RdBu_r",  r"Vorticity $\omega$"),
    ("phi",       phi,      "RdBu_r",  r"Potential $\phi$"),
]

fig, axes = plt.subplots(len(FIELDS), N_SNAPS,
                         figsize=(3.5 * N_SNAPS, 3.2 * len(FIELDS)))

for row, (name, data, cmap, label) in enumerate(FIELDS):
    for col, idx in enumerate(snap_idxs):
        ax  = axes[row, col]
        frame = data[idx]
        vmax = np.percentile(np.abs(frame), 99)
        im = ax.imshow(frame, origin="lower", cmap=cmap,
                       vmin=-vmax, vmax=vmax, aspect="equal")
        ax.set_xticks([]); ax.set_yticks([])
        if col == 0:
            ax.set_ylabel(label, fontsize=11)
        if row == 0:
            ax.set_title(f"t = {times[idx]:.1f}", fontsize=10)
        plt.colorbar(im, ax=ax, fraction=0.046, pad=0.02)

fig.suptitle("HW2D 2D field snapshots", fontsize=13, y=1.01)
fig.tight_layout()
snap_fig_path = OUTPUT_DIR / "hw2d_snapshots.png"
fig.savefig(snap_fig_path, dpi=130, bbox_inches="tight")
print(f"Saved: {snap_fig_path}")
plt.show()

## 4 — Animated Video

In [ ]:
# Build the frame index array
frame_idxs = np.arange(drop, T, ANIM_FRAME_STRIDE)
print(f"Animating {len(frame_idxs)} frames at stride={ANIM_FRAME_STRIDE}")

# Pre-compute symmetric colour limits (99th percentile of each field over all frames)
def _sym_lim(arr, pct=99):
    return float(np.percentile(np.abs(arr[drop::ANIM_FRAME_STRIDE]), pct))

vlim = {
    "density": _sym_lim(density),
    "omega":   _sym_lim(omega),
    "phi":     _sym_lim(phi),
}

# ── Set up figure ─────────────────────────────────────────────────────────────
fig_anim, axs = plt.subplots(1, 3, figsize=(13, 4.5))
fig_anim.patch.set_facecolor("#111")
for ax in axs:
    ax.set_facecolor("#111")

anim_fields = [
    (axs[0], density, "density", r"Density $n$"),
    (axs[1], omega,   "omega",   r"Vorticity $\omega$"),
    (axs[2], phi,     "phi",     r"Potential $\phi$"),
]

ims, cbars = [], []
for ax, data, key, label in anim_fields:
    vmax = vlim[key]
    im = ax.imshow(data[frame_idxs[0]], origin="lower",
                   cmap="RdBu_r", vmin=-vmax, vmax=vmax, aspect="equal")
    ax.set_title(label, color="white", fontsize=11)
    ax.set_xticks([]); ax.set_yticks([])
    cb = fig_anim.colorbar(im, ax=ax, fraction=0.046, pad=0.02)
    cb.ax.yaxis.set_tick_params(color="white")
    plt.setp(cb.ax.yaxis.get_ticklabels(), color="white")
    ims.append(im)
    cbars.append(cb)

time_label = fig_anim.text(
    0.5, 1.00, "", ha="center", va="top",
    color="white", fontsize=12, transform=fig_anim.transFigure
)
fig_anim.tight_layout()

def _update(i):
    idx = frame_idxs[i]
    ims[0].set_data(density[idx])
    ims[1].set_data(omega[idx])
    ims[2].set_data(phi[idx])
    time_label.set_text(f"t = {times[idx]:.2f}")
    return ims + [time_label]

ani = animation.FuncAnimation(
    fig_anim, _update,
    frames=len(frame_idxs),
    interval=1000 / VIDEO_FPS,
    blit=True,
)

plt.close(fig_anim)  # don't show the static figure — display via HTML below
print("Animation built.")

In [ ]:
# ── Display inline in notebook ────────────────────────────────────────────────
# (this renders an interactive HTML5 player; may take a moment to encode)
display(HTML(ani.to_jshtml(fps=VIDEO_FPS)))

In [ ]:
# ── Save video to disk ────────────────────────────────────────────────────────
if SAVE_VIDEO:
    suffix = VIDEO_PATH.suffix.lower()

    if suffix == ".mp4":
        try:
            writer = animation.FFMpegWriter(
                fps=VIDEO_FPS, codec="h264",
                extra_args=["-pix_fmt", "yuv420p"]   # broad compatibility
            )
            ani.save(str(VIDEO_PATH), writer=writer, dpi=VIDEO_DPI)
            print(f"Saved MP4: {VIDEO_PATH}  ({VIDEO_PATH.stat().st_size / 1e6:.1f} MB)")
        except Exception as e:
            print(f"ffmpeg not available ({e}) — falling back to GIF")
            gif_path = VIDEO_PATH.with_suffix(".gif")
            writer_gif = animation.PillowWriter(fps=VIDEO_FPS)
            ani.save(str(gif_path), writer=writer_gif, dpi=VIDEO_DPI)
            print(f"Saved GIF: {gif_path}  ({gif_path.stat().st_size / 1e6:.1f} MB)")

    elif suffix == ".gif":
        writer_gif = animation.PillowWriter(fps=VIDEO_FPS)
        ani.save(str(VIDEO_PATH), writer=writer_gif, dpi=VIDEO_DPI)
        print(f"Saved GIF: {VIDEO_PATH}  ({VIDEO_PATH.stat().st_size / 1e6:.1f} MB)")

    else:
        print(f"Unknown format '{suffix}' — set VIDEO_PATH to .mp4 or .gif")
else:
    print("SAVE_VIDEO=False — skipping save")

## 5 — Bonus: Energy Spectrum

In [ ]:
# Radially-averaged kinetic energy spectrum at a late frame
late_idx = frame_idxs[-1]
phi_frame = phi[late_idx]
Ny, Nx = phi_frame.shape

# Fourier-based kinetic energy: E(k) ~ k^2 |phi_k|^2
phi_k  = np.fft.fft2(phi_frame)
kx     = np.fft.fftfreq(Nx, d=1.0 / Nx)
ky     = np.fft.fftfreq(Ny, d=1.0 / Ny)
KX, KY = np.meshgrid(kx, ky)
K      = np.sqrt(KX**2 + KY**2)
Ek     = (K**2) * np.abs(phi_k)**2

# Radial binning
k_bins = np.arange(0.5, min(Nx, Ny) // 2)
k_mids = 0.5 * (k_bins[:-1] + k_bins[1:])
spec   = np.array([
    Ek[(K >= k_bins[i]) & (K < k_bins[i+1])].mean()
    for i in range(len(k_bins) - 1)
])

fig, ax = plt.subplots(figsize=(7, 4))
ax.loglog(k_mids, spec, lw=1.5, color="steelblue", label="KE spectrum")
# Reference slopes
k_ref = k_mids[5:20]
ax.loglog(k_ref, spec[5] * (k_ref / k_ref[0])**(-3),
          "--", color="gray", lw=1, label=r"$k^{-3}$")
ax.loglog(k_ref, spec[5] * (k_ref / k_ref[0])**(-5/3),
          ":",  color="tomato", lw=1, label=r"$k^{-5/3}$")
ax.set_xlabel("Wavenumber $k$")
ax.set_ylabel(r"$E(k)$")
ax.set_title(f"Kinetic energy spectrum at t = {times[late_idx]:.1f}")
ax.legend()
fig.tight_layout()
spec_path = OUTPUT_DIR / "hw2d_spectrum.png"
fig.savefig(spec_path, dpi=130, bbox_inches="tight")
print(f"Saved: {spec_path}")
plt.show()